In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv('../data/heart_disease.csv')

In [3]:
df.head()

,Age,Gender,Cholesterol,Blood Pressure,Heart Rate,Smoking,Alcohol Intake,Exercise Hours,Family History,Diabetes,Obesity,Stress Level,Blood Sugar,Exercise Induced Angina,Chest Pain Type,Heart Disease
0,75,Female,228,119,66,Current,Heavy,1,No,No,Yes,8,119,Yes,Atypical Angina,1
1,48,Male,204,165,62,Current,NaN,5,No,No,No,9,70,Yes,Typical Angina,0
2,53,Male,234,91,67,Never,Heavy,3,Yes,No,Yes,5,196,Yes,Atypical Angina,1
3,69,Female,192,90,72,Current,NaN,4,No,Yes,No,7,107,Yes,Non-anginal Pain,0
4,62,Female,172,163,93,Never,NaN,6,No,Yes,No,2,183,Yes,Asymptomatic,0


In [4]:
X = df.drop('Heart Disease',axis = 1)
y = df['Heart Disease']

In [5]:
numerical_features = X.select_dtypes(include=["int64","float64"]).columns
categorical_features = X.select_dtypes(include=["str"]).columns

In [6]:
from sklearn.model_selection import train_test_split

X_train,X_test,y_train,y_test = train_test_split(
     X,
     y,
     stratify=y,
     random_state=42,
     test_size=0.20
 )


In [7]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler , OneHotEncoder

preprocessor = ColumnTransformer(
    transformers = [
        (
            "num",
            Pipeline(
                [
                    ("imputer",SimpleImputer(strategy="median")),
                    ("scaler",StandardScaler())
                ]
            ),
            numerical_features
        ),
         (
                "cat",
            Pipeline(
                [
                    ("imputer",SimpleImputer(strategy="most_frequent")),
                    ("scaler",OneHotEncoder(handle_unknown="ignore"))
                ]
            ),
            categorical_features
        )
    ]
)

X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

In [9]:
from sklearn.model_selection import GridSearchCV
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC

knn = KNeighborsClassifier()
svc_model = SVC()

param_grid = {
    "n_neighbors":[1,3,5,7,9,11,13,15,17,19,21],
    "weights":["uniform","distance"]
}

grid_search = GridSearchCV(
    estimator=knn,
    param_grid = param_grid,
    cv=5,
    scoring="accuracy"
)

grid_search.fit(X_train_processed,y_train)

y_pred = grid_search.predict(X_test_processed)

print("Best Parameters:", grid_search.best_params_)
print("Best CV Score:", grid_search.best_score_)

Best Parameters: {'n_neighbors': 19, 'weights': 'uniform'}
Best CV Score: 0.9012499999999999
